# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm choosing this lane because it ends in something a real person can act on this week, not just a chart. The starter pipeline already shows that a learned ranking clearly beats a fixed rule on this exact data (see the numbers below), which tells me there's real, messy-but-learnable signal here — the kind of problem ML is actually for, instead of an if-statement. It also lines up with my own interest in practical, deploy-able tooling rather than pure analysis: a ranked review queue is something a small team could genuinely use. I'm not locked in — I can confirm or swap this by the end of Week 4 — but it's my starting bet.

In [1]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(df.shape, '| clients:', df['client_id'].nunique())
df.head(3)

(30000, 44) | clients: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The question:** Given a content team with limited review time, which pages in the inventory should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

**Unit of analysis:** one content item (one page, identified by `content_id`) evaluated over its trailing-90-day window.

**Output:** a ranked queue of pages, each with a priority score and a short reason code (e.g. "stale and still visible," "declining with demand," "thin but getting impressions") that a human can sanity-check in seconds.

**Who acts, and how:** a content editor or SEO strategist with a fixed number of review slots per week. Instead of scrolling an unordered spreadsheet of thousands of pages, they open the top of the queue and start there.

**The decision this improves:** *which page to spend a reviewer's next hour on*, not "will this page decline" in the abstract. That's a ranking decision, not a yes/no prediction.

**Cost of a wrong call:**
- *False positive* (a page ranked high that didn't need attention): a wasted review hour — annoying, but cheap, and the reviewer notices quickly and moves on.
- *False negative* (a real opportunity buried low in the queue): a genuinely declining or under-performing page never gets looked at, and the traffic/revenue behind it keeps leaking — this is the costlier direction, since limited review capacity means low-ranked pages may simply never be seen.
- Because false negatives are quieter and more expensive than false positives, I'll care more about precision in the *top* of the queue (precision@K) than about overall accuracy.

**Why data/ML helps at all (and why not just an if-statement):** a simple rule (e.g. "old + high impressions") already exists and is cheap to compute — I'll build that first as my honest baseline. But real pages fail for tangled, overlapping reasons (stale *and* thin *and* losing rank *and* fine on CTR), and the relative importance of those signals isn't obvious by hand. The starter pipeline in this repo already tested this: a plain rule-based baseline scores ROC-AUC 0.627 / Precision@50 0.240, while a random forest trained on the same signals reaches ROC-AUC 0.750 / Precision@50 0.740 — meaning roughly 37 of the top 50 flagged pages are right, versus about 12 for the rule. That gap is the evidence that a model earns its place here instead of a hand-written if-statement.

In [2]:
# Verified from outputs/model_report.md (committed starter run) — quoting the headline comparison here:
comparison = pd.DataFrame({
    'method': ['baseline rules', 'logistic regression', 'decision tree', 'random forest'],
    'roc_auc': [0.627, 0.700, 0.742, 0.750],
    'avg_precision': [0.468, 0.522, 0.575, 0.618],
    'precision_at_50': [0.240, 0.400, 0.540, 0.740],
})
comparison['hits_in_top_50'] = (comparison['precision_at_50'] * 50).round(0).astype(int)
comparison

,method,roc_auc,avg_precision,precision_at_50,hits_in_top_50
0,baseline rules,0.627,0.468,0.24,12
1,logistic regression,0.700,0.522,0.40,20
2,decision tree,0.742,0.575,0.54,27
3,random forest,0.750,0.618,0.74,37


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

I already loaded the CSV above (30,000 rows, 32 clients). Here I check how big the candidate pool actually is using the exact reason-code rules from the lane guide — the same rules the starter baseline uses.

In [3]:
n = len(df)

stale_visible_page = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)
declining_with_demand = (df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)
thin_visible_page = (df['word_count'] > 0) & (df['word_count'] < 1200) & (df['impressions_90d'] >= 250)
page_one_decay_risk = (df['avg_position'] > 0) & (df['avg_position'] <= 10) & (df['content_age_days'] >= 180)
low_ctr_visible_page = (df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)

any_flag = stale_visible_page | declining_with_demand | thin_visible_page | page_one_decay_risk | low_ctr_visible_page

print(f"Rows total: {n:,}")
print(f"declining_with_demand: {declining_with_demand.sum():,} ({declining_with_demand.mean()*100:.1f}%)")
print(f"page_one_decay_risk:   {page_one_decay_risk.sum():,} ({page_one_decay_risk.mean()*100:.1f}%)")
print(f"low_ctr_visible_page:  {low_ctr_visible_page.sum():,} ({low_ctr_visible_page.mean()*100:.1f}%)")
print(f"ANY reason code fires: {any_flag.sum():,} ({any_flag.mean()*100:.1f}%) of pages")
print()
print("avg_position == 0 (no rank data, must be excluded from position logic):", (df['avg_position'] == 0).sum())

Rows total: 30,000
declining_with_demand: 13,152 (43.8%)
page_one_decay_risk:   7,076 (23.6%)
low_ctr_visible_page:  9,759 (32.5%)
ANY reason code fires: 19,650 (65.5%) of pages

avg_position == 0 (no rank data, must be excluded from position logic): 1205


**Reading these numbers:**

- **43.8%** of the 30,000 pages (13,152) are flagged `declining_with_demand` — a huge pool, which is exactly the problem: a reviewer cannot look at 13,000 pages, they need them *ordered*.
- **65.5%** of all pages (19,650) trip at least one reason code — so a plain rulebook alone doesn't narrow the queue much; most of the work is in ranking *within* that flagged set, which is where a model's continuous score beats a rule's yes/no flag.
- **1,205 rows** have `avg_position == 0`, which means "no rank data" and not "rank zero" — a reminder I have to handle that as a missingness flag, not a real position, when I build features later.

Together these numbers say the candidate pool is large and the rules alone don't discriminate well within it — which is the setup where a learned ranking (Lane 2) has room to add real value, matching the baseline-vs-model gap shown in section 2.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- **Observed / measured** results: e.g. "X% of pages in this 90-day window show pattern Y," backed by a number I computed from the data.
- **Directional / associational** findings: e.g. "pages with signal A tend to also show weakness B" — a pattern, not a law.
- **Decision-support** output: a ranked queue that helps a human prioritize limited review time. The ranking is a recommendation for *where to look first*, not a guarantee of what they'll find.

**What I will never claim:**
- **Causal proof.** I cannot say "refreshing this page will cause it to recover" — that needs an experiment (before/after with a control), which this dataset doesn't give me. At most I can say a page matches the profile of pages that later recovered *when reviewed*, if I can even establish that pattern.
- **Anything about Google's actual ranking algorithm.** I'm working with observed search/analytics signals, not the algorithm itself.
- **That a "declining" label is destiny.** Section 7 of the lane guide is explicit that real decline has look-alikes — consolidation (a sibling page absorbed the traffic), seasonality, SERP/AI layout changes, and plain noise. Before I call anything "declining" I'll need to rule these out with magnitude, window, persistence, and minimum-volume checks — not just point to a negative `trend_pct`.
- **That the model's top pick is always right.** Precision@50 of 0.740 means roughly 13 of the top 50 are still *not* what the label says — good enough to prioritize a queue, nowhere near good enough to skip human judgment.

In [4]:
# No additional computation needed for this section — the caveats above are qualitative commitments,
# not numbers to derive. Leaving this cell as a placeholder per the skeleton structure.
print("Section 4 is a written commitment, not a computed result — see the markdown above.")

Section 4 is a written commitment, not a computed result — see the markdown above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.